In [ ]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from agent_lab.model_hub import LLM_FAST

model = init_chat_model(**LLM_FAST)

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [2]:
# Step 1: Model generates tool calls
messages = [HumanMessage("What's the weather in Boston?")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)  # type: ignore
ai_msg.content_blocks

[{'type': 'tool_call',
  'id': 'call_00_BGpBDimxkQuEbDHVJ8IrEyKe',
  'name': 'get_weather',
  'args': {'location': 'Boston'}}]

In [3]:
ai_msg.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Boston'},
  'id': 'call_00_BGpBDimxkQuEbDHVJ8IrEyKe',
  'type': 'tool_call'}]

In [4]:
# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    tool_result.pretty_print()

================================= Tool Message =================================
Name: get_weather

It's sunny in Boston.


In [5]:
# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
final_response.pretty_print()

================================== Ai Message ==================================

The weather in Boston is currently **sunny**! ☀️


#### 直接调用工具函数

In [6]:
tool_call

{'name': 'get_weather',
 'args': {'location': 'Boston'},
 'id': 'call_00_BGpBDimxkQuEbDHVJ8IrEyKe',
 'type': 'tool_call'}

In [7]:
get_weather.invoke(tool_call)

ToolMessage(content="It's sunny in Boston.", name='get_weather', tool_call_id='call_00_BGpBDimxkQuEbDHVJ8IrEyKe')

In [8]:
get_weather.run(tool_call['args'])

"It's sunny in Boston."